In [1]:
from pathlib import Path
import re

___
### Extracting useful info from log file

Log file contains lot of metadata, such as date, time, process id at the beginning of each line. This is unnecessary for our data<br><br>
We only need lines with block id information, and the message in the line to build template<br><br>
Example line from the log:<br>
081111 041138 19464 INFO dfs.DataNodeDataXceiver: Receiving block blk_-4750397141673817648 src: /10.251.70.112:52693 dest: /10.251.70.112:50010<br><br>
Here, "081111 041138 19464 INFO dfs.DataNodeDataXceiver" is unnecessary. We only need "blk_-4750397141673817648", and "Receiving block blk_-4750397141673817648 src:(.) dest: (.) <br><br>

Regex for block id: (blk_-?\d+)<br>
&ensp;blk_  -  must be present<br>
&ensp;-?    -  '-' is optional<br>
&ensp;\d+   -   any number of digits after that is acceptable<br><br>

Parsing message:<br>
From sample line, we understand that message part lies after the first colon. Hence, split line based on first colon, then second part of sentence will be message. Message still not cleaned properly at this stage, as things like src and dest will still have numbers

In [4]:
log_path = Path('../datasets/HDFS_v1/HDFS.log')

def find_pattern(log_path): 
    data = []
    
    blk_pattern = re.compile(r'(blk_-?\d+)')

    with open(log_path) as f:
        for line in f:
            blk_match = blk_pattern.search(line)
            if not blk_match:
                continue

            blk_id = blk_match.group(1)

            parts = line.split(':', maxsplit=1)
            if len(parts) < 2:
                continue
            msg = parts[1].strip()

            data.append((blk_id, msg))

    return data

In [9]:
parsed_data = find_pattern(log_path)
print('No of lines parsed: ', len(parsed_data))

No of lines parsed:  11175629


In [12]:
import numpy as np
print('5 sample lines: ')
for i in range(5):
    rand = np.random.randint(0, len(parsed_data))
    print(parsed_data[rand])

5 sample lines: 
('blk_-165409665468807885', 'BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.201.204:50010 is added to blk_-165409665468807885 size 67108864')
('blk_2204439492614869623', 'Deleting block blk_2204439492614869623 file /mnt/hadoop/dfs/data/current/subdir53/blk_2204439492614869623')
('blk_7408352197483159129', 'Deleting block blk_7408352197483159129 file /mnt/hadoop/dfs/data/current/subdir58/blk_7408352197483159129')
('blk_1411954036626688278', 'BLOCK* NameSystem.delete: blk_1411954036626688278 is added to invalidSet of 10.251.67.211:50010')
('blk_6491734079886303038', 'PacketResponder 1 for block blk_6491734079886303038 terminating')


___

### Normalising message to follow template structure
<br>
Messages(events) have been extracted, but they contain blk_id, source and destination IPs, size etc, which vary from line to line, even if the event is the same. This would cause the each line to be classfied as different event<br>
Hence, next step:<br><br>

Normalise event messages<br>
Ex: Convert 'Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010' &ensp; TO &ensp; 'Receiving block (.) src: (.) dest: (.)'<br><br>

Regex used:<br>
&ensp; subbing "blk_-?\d+" ----- Subbing blk_id<br>
&ensp; subbing "/?\d+\.\d+\.\d+\.\d+(?::\d+)?:?" ----- Subbing IPs and ports<br>
&ensp; subbing "\b\d+\b" ----- Subbing standalone numbers such as size<br><br>

Noticed that one more regex subbing has to be applied for treating directories, like in cases of:<br><br>
('blk_-3100963327775157803', 'Deleting block <> file /mnt/hadoop/dfs/data/current/subdir16/<>')<br>
('blk_7185070990871707682', 'Deleting block <> file /mnt/hadoop/dfs/data/current/subdir50/<>')<br>
The paths in these two examples will cause the two to be separate templates, although they are evidently performing the same operation<br><br>

Solution for paths:<br>
&ensp; subbing "/[^\s]+" ---- removes anything that starts with a / (mostly paths)

In [23]:
def normalise_message(msg):
    msg = re.sub(r"blk_-?\d+", "<*>", msg) # Removing blk_id
    msg = re.sub(r"/?\d+\.\d+\.\d+\.\d+(?::\d+)?:?", "<*>", msg) # Removing IP addresses and ports
    msg = re.sub(r"\b\d+\b", "<*>", msg) # Removing standalone numbers like size
    msg = re.sub(r"/[^\s]+", "<*>", msg) # Removing paths 

    return msg

In [24]:
# Im saving under same variable name to save memory
parsed_data = [
    (blk_id, normalise_message(msg)) for blk_id, msg in parsed_data
]

In [28]:
print('5 Random normalised data: ')
for i in range(5):
    rand = np.random.randint(0, len(parsed_data))
    print(parsed_data[rand])

5 Random normalised data: 
('blk_7521017274574722666', 'Receiving block <*> src: <*> dest: <*>')
('blk_5951940314446438918', '<*> Served block <*> to <*>')
('blk_-7209808647878157356', 'BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>')
('blk_1056899005864211225', '<*> Served block <*> to <*>')
('blk_4162832865986721185', '<*>Got exception while serving <*> to <*>')
